<a href="https://colab.research.google.com/github/ArkanUbaidillah/BigData26_B_2411537001_ArkanUbaidillahWarman/blob/main/Praktikum01/BD_B_P01_2411537001_ArkanUbaidillahWarman.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Praktikum 1 — Pengenalan Big Data & Lingkungan Kerja, serta Pengolahan Dataset Big Data Menggunakan Python

**Mata kuliah:** Praktikum Big Data · **Prodi/Semester:** S1 Informatika / Semester V
**Nama file:** `BD_B_P01_2411537001_ArkanUbaidillahWarman.ipynb` · **Platform:** Google Colab (runtime CPU)
**Dataset:** NYC TLC Yellow Taxi Trip Records, `yellow_tripdata_2023-01.parquet`


## J-1. Verifikasi lingkungan kerja

Semua kesimpulan performa pada praktikum ini hanya berlaku untuk spesifikasi mesin yang tercetak di bawah, karena itu versi Python, pustaka, RAM, disk, dan Java harus diketahui persis lebih dulu.

In [78]:
import sys, platform, multiprocessing

print("Python  :", sys.version.split()[0])
print("OS      :", platform.platform())
print("CPU core:", multiprocessing.cpu_count())

for nama in ["pandas", "numpy", "pyarrow", "matplotlib"]:
    try:
        mod = __import__(nama)
        print(f"{nama:11s}: {mod.__version__}")
    except ImportError:
        print(f"{nama:11s}: BELUM TERPASANG")

Python  : 3.13.15
OS      : Linux-6.6.122+-x86_64-with-glibc2.39
CPU core: 2
pandas     : 2.2.3
numpy      : 2.1.3
pyarrow    : 23.0.1
matplotlib : 3.10.0


In [79]:
!free -h             # kapasitas dan sisa RAM
!df -h /content      # kapasitas dan sisa disk runtime
!java -version       # dibutuhkan PySpark pada langkah K-8

               total        used        free      shared  buff/cache   available
Mem:            12Gi       4.7Gi       4.2Gi       2.9Mi       4.1Gi       8.0Gi
Swap:             0B          0B          0B
Filesystem      Size  Used Avail Use% Mounted on
overlay         108G   22G   87G  21% /
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
openjdk version "21.0.12" 2026-07-21
OpenJDK Runtime Environment (build 21.0.12+8-1-24.04-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.12+8-1-24.04-Ubuntu, mixed mode, sharing)


## J-2. Menghubungkan Google Drive dan menyiapkan struktur folder

File di `/content` bersifat sementara dan hilang saat runtime berakhir, sehingga semua artefak penting harus berakhir di Google Drive. Sel ini memasang Drive dan membuat folder kerja sementara serta folder simpan permanen.

In [80]:
from google.colab import drive
drive.mount("/content/drive")          # ikuti dialog izin yang muncul

import os
DIR_KERJA  = "/content/data"                               # sementara, cepat
DIR_SIMPAN = "/content/drive/MyDrive/BigData/Praktikum1"   # permanen
os.makedirs(DIR_KERJA, exist_ok=True)
os.makedirs(DIR_SIMPAN, exist_ok=True)
print(os.listdir(DIR_SIMPAN))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['agregasi_per_jam.csv', 'agregasi_per_jam.parquet', 'pengukuran_kinerja.csv', 'distribusi_per_jam.png', 'studi_kasus_pola_hari.png', 'studi_kasus_tarif_per_menit.png', 'agregasi_per_jam_2023-07.csv', 'agregasi_per_jam_2023-07.parquet', 'perbandingan_dua_bulan.png']


## J-3. Menyiapkan alat ukur waktu dan memori

Praktikum ini menuntut bukti angka, bukan kesan. Fungsi `ukur()` mencatat durasi dan pertumbuhan RSS (memori fisik proses) setiap langkah ke dalam list `catatan` agar bisa dibandingkan nanti.

In [81]:
import time, os, psutil

proses = psutil.Process(os.getpid())
catatan = []            # menyimpan hasil pengukuran untuk dibandingkan nanti

def rss_mb():
    return proses.memory_info().rss / 1024**2

def ukur(label, fungsi):
    """Menjalankan fungsi tanpa argumen, mencatat durasi dan pertumbuhan memori."""
    m0, t0 = rss_mb(), time.perf_counter()
    hasil  = fungsi()
    detik  = time.perf_counter() - t0
    delta  = rss_mb() - m0
    catatan.append({"langkah": label, "detik": round(detik, 2),
                    "delta_rss_mb": round(delta, 1)})
    print(f"[{label}] {detik:.2f} s | RSS {delta:+.1f} MB")
    return hasil

## J-4. Mengunduh dataset dan inspeksi awal

Dataset diunduh terprogram dari sumber resmi NYC TLC, lalu skema dan jumlah barisnya dibaca dari metadata Parquet tanpa memuat satu baris pun ke memori.

In [82]:
import urllib.request

URL  = ("https://d37ci6vzurychx.cloudfront.net/trip-data/"
        "yellow_tripdata_2023-01.parquet")
PATH = os.path.join(DIR_KERJA, "yellow_tripdata_2023-01.parquet")

if not os.path.exists(PATH):
    ukur("unduh dataset", lambda: urllib.request.urlretrieve(URL, PATH))

print("Ukuran file:", round(os.path.getsize(PATH) / 1024**2, 1), "MB")

Ukuran file: 45.5 MB


In [83]:
import pyarrow.parquet as pq

meta = pq.ParquetFile(PATH).metadata
print("Jumlah baris  :", f"{meta.num_rows:,}")
print("Jumlah kolom  :", meta.num_columns)
print("Row group     :", meta.num_row_groups)
print("Nama kolom    :", pq.ParquetFile(PATH).schema.names)

Jumlah baris  : 3,066,766
Jumlah kolom  : 19
Row group     : 1
Nama kolom    : ['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'airport_fee']


## J-4b. Jalur cadangan bila unduhan gagal

Bila jaringan memblokir unduhan, sel ini membangkitkan dataset sintetis 5 juta baris dengan skema serupa. Pada eksekusi ini unduhan berhasil, sehingga sel ini tidak berbuat apa-apa (guard `if not os.path.exists(PATH)`).

In [84]:
import numpy as np, pandas as pd

def bangkitkan_sintetis(n=5_000_000, seed=42):
    rng   = np.random.default_rng(seed)
    awal  = pd.Timestamp("2023-01-01")
    pickup = awal + pd.to_timedelta(rng.integers(0, 31*24*60, n), unit="m")
    durasi = rng.integers(1, 90, n)
    return pd.DataFrame({
        "tpep_pickup_datetime":  pickup,
        "tpep_dropoff_datetime": pickup + pd.to_timedelta(durasi, unit="m"),
        "passenger_count":       rng.integers(1, 5, n).astype("float64"),
        "trip_distance":         np.round(rng.gamma(2.0, 1.6, n), 2),
        "payment_type":          rng.integers(1, 5, n).astype("int64"),
        "fare_amount":           np.round(rng.gamma(4.0, 3.0, n), 2),
        "tip_amount":            np.round(rng.gamma(1.5, 1.2, n), 2),
    }).assign(total_amount=lambda d: (d.fare_amount + d.tip_amount + 3.0).round(2))

if not os.path.exists(PATH):
    bangkitkan_sintetis().to_parquet(PATH, index=False)
    print("Dataset sintetis dibuat di", PATH)

## J-5. Memuat data dan mengukur biaya memorinya

Dua percobaan dibandingkan: memuat semua kolom versus memuat hanya kolom yang dibutuhkan, lalu pemakaian memori dipersempit lagi dengan tipe data yang lebih kecil.

In [85]:
import pandas as pd

KOLOM = ["tpep_pickup_datetime", "tpep_dropoff_datetime",
         "passenger_count", "trip_distance", "payment_type",
         "fare_amount", "tip_amount", "total_amount"]

penuh = ukur("baca semua kolom", lambda: pd.read_parquet(PATH))
print("Dimensi:", penuh.shape)
print("Memori :", round(penuh.memory_usage(deep=True).sum() / 1024**2, 1), "MB")

del penuh                              # bebaskan memori sebelum percobaan kedua
import gc; gc.collect()

df = ukur("baca kolom terpilih", lambda: pd.read_parquet(PATH, columns=KOLOM))
print("Dimensi:", df.shape)
print("Memori :", round(df.memory_usage(deep=True).sum() / 1024**2, 1), "MB")
df.head()

[baca semua kolom] 1.63 s | RSS +443.4 MB
Dimensi: (3066766, 19)
Memori : 565.6 MB
[baca kolom terpilih] 0.37 s | RSS +163.8 MB
Dimensi: (3066766, 8)
Memori : 187.2 MB


,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,payment_type,fare_amount,tip_amount,total_amount
0,2023-01-01 00:32:10,2023-01-01 00:40:36,1.0,0.97,2,9.3,0.00,14.30
1,2023-01-01 00:55:08,2023-01-01 01:01:27,1.0,1.10,1,7.9,4.00,16.90
2,2023-01-01 00:25:04,2023-01-01 00:37:49,1.0,2.51,1,14.9,15.00,34.90
3,2023-01-01 00:03:48,2023-01-01 00:13:25,0.0,1.90,1,12.1,0.00,20.85
4,2023-01-01 00:10:29,2023-01-01 00:21:19,1.0,1.43,1,11.4,3.28,19.68


In [86]:
df.info(memory_usage="deep")
df.memory_usage(deep=True).sort_values(ascending=False) / 1024**2    # MB per kolom

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3066766 entries, 0 to 3066765
Data columns (total 8 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   tpep_pickup_datetime   datetime64[us]
 1   tpep_dropoff_datetime  datetime64[us]
 2   passenger_count        float64       
 3   trip_distance          float64       
 4   payment_type           int64         
 5   fare_amount            float64       
 6   tip_amount             float64       
 7   total_amount           float64       
dtypes: datetime64[us](2), float64(5), int64(1)
memory usage: 187.2 MB


,0
tpep_pickup_datetime,23.397568
payment_type,23.397568
tpep_dropoff_datetime,23.397568
passenger_count,23.397568
trip_distance,23.397568
tip_amount,23.397568
fare_amount,23.397568
total_amount,23.397568
Index,0.000126


In [87]:
sebelum = df.memory_usage(deep=True).sum() / 1024**2

df["passenger_count"] = pd.to_numeric(df["passenger_count"], downcast="float")
df["payment_type"]    = df["payment_type"].astype("int8").astype("category")
for kol in ["trip_distance", "fare_amount", "tip_amount", "total_amount"]:
    df[kol] = pd.to_numeric(df[kol], downcast="float")

sesudah = df.memory_usage(deep=True).sum() / 1024**2
print(f"{sebelum:.1f} MB -> {sesudah:.1f} MB  (hemat {100*(1-sesudah/sebelum):.1f}%)")
df.dtypes

187.2 MB -> 119.9 MB  (hemat 35.9%)


,0
tpep_pickup_datetime,datetime64[us]
tpep_dropoff_datetime,datetime64[us]
passenger_count,float32
trip_distance,float64
payment_type,category
fare_amount,float32
tip_amount,float32
total_amount,float32


## J-6. Pemeriksaan kualitas data dan pembersihan (veracity)

Durasi perjalanan diturunkan dari selisih waktu drop-off dan pickup, lalu anomali (jarak nol, durasi negatif atau tidak masuk akal, tarif minus) dihitung dan disaring. Setiap threshold adalah keputusan analitis yang alasannya ditulis di laporan.

In [88]:
df["durasi_menit"] = (df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]).dt.total_seconds() / 60

print(df[["trip_distance", "durasi_menit", "total_amount"]].describe())
print("\nNilai kosong per kolom:\n", df.isna().sum())

anomali = {
    "jarak <= 0"         : (df["trip_distance"] <= 0).sum(),
    "durasi <= 0"        : (df["durasi_menit"]  <= 0).sum(),
    "durasi > 180 menit" : (df["durasi_menit"]  > 180).sum(),
    "total_amount <= 0"  : (df["total_amount"]  <= 0).sum(),
}
for k, v in anomali.items():
    print(f"{k:22s}: {v:,} baris ({100*v/len(df):.2f}%)")

       trip_distance  durasi_menit  total_amount
count   3.066766e+06  3.066766e+06  3.066766e+06
mean    3.847342e+00  1.566900e+01  2.702038e+01
std     2.495838e+02  4.259435e+01  2.212666e+01
min     0.000000e+00 -2.920000e+01 -7.510000e+02
25%     1.060000e+00  7.116667e+00  1.540000e+01
50%     1.800000e+00  1.151667e+01  2.016000e+01
75%     3.330000e+00  1.830000e+01  2.870000e+01
max     2.589281e+05  1.002918e+04  1.169400e+03

Nilai kosong per kolom:
 tpep_pickup_datetime         0
tpep_dropoff_datetime        0
passenger_count          71743
trip_distance                0
payment_type                 0
fare_amount                  0
tip_amount                   0
total_amount                 0
durasi_menit                 0
dtype: int64
jarak <= 0            : 45,862 baris (1.50%)
durasi <= 0           : 1,121 baris (0.04%)
durasi > 180 menit    : 3,048 baris (0.10%)
total_amount <= 0     : 25,772 baris (0.84%)


In [89]:
layak = (
    (df["trip_distance"] > 0) & (df["trip_distance"] < 100) &
    df["durasi_menit"].between(1, 180) &
    (df["total_amount"] > 0)
)
bersih = df.loc[layak].copy()
print(f"{len(df):,} baris -> {len(bersih):,} baris   (dibuang {len(df)-len(bersih):,})")

3,066,766 baris -> 2,986,911 baris   (dibuang 79,855)


## J-7. Memproses file lebih besar dari memori dengan chunking

Kondisi klasik "file terlalu besar" disimulasikan dengan menulis ulang data sebagai CSV (teks tanpa kompresi, jauh lebih besar), lalu file itu diagregasi bagian demi bagian tanpa pernah dimuat utuh.

In [ ]:
CSV = os.path.join(DIR_KERJA, "trips.csv")
ukur("tulis CSV", lambda: bersih.to_csv(CSV, index=False))

print("Parquet:", round(os.path.getsize(PATH) / 1024**2, 1), "MB")
print("CSV    :", round(os.path.getsize(CSV)  / 1024**2, 1), "MB")

In [ ]:
def agregasi_bertahap(path, ukuran_chunk=500_000):
    jumlah_baris = 0
    total_nilai  = 0.0
    per_jam      = {}
    potongan = pd.read_csv(
        path,
        usecols=["tpep_pickup_datetime", "total_amount"],
        parse_dates=["tpep_pickup_datetime"],
        chunksize=ukuran_chunk)
    for chunk in potongan:
        jumlah_baris += len(chunk)
        total_nilai  += chunk["total_amount"].sum()
        for jam, sub in chunk.groupby(chunk["tpep_pickup_datetime"].dt.hour):
            n, nilai = per_jam.get(jam, (0, 0.0))
            per_jam[jam] = (n + len(sub), nilai + sub["total_amount"].sum())
    return jumlah_baris, total_nilai, per_jam

baris, total, per_jam_chunk = ukur("agregasi chunking", lambda: agregasi_bertahap(CSV))
print(f"{baris:,} baris | rata-rata total_amount = {total/baris:.2f}")

## J-8. Pendekatan terdistribusi: PySpark local mode

PySpark 3.5.1 dipasang dan dijalankan pada semua core mesin ini (`local[*]`) untuk memahami konsep lazy evaluation, partisi, dan shuffle tanpa klaster. Untuk data sebesar ini pandas diperkirakan tetap lebih cepat, dan itu bukan kegagalan Spark.

In [ ]:
!pip install -q pyspark==3.5.1
!java -version

In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
         .appName("BD-P01")
         .master("local[*]")                # semua core mesin ini
         .config("spark.driver.memory", "4g")
         .config("spark.sql.shuffle.partitions", "8")
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")
print("Spark:", spark.version)

In [ ]:
sdf = spark.read.parquet(PATH)
sdf.printSchema()
print("Jumlah baris:", ukur("spark count", lambda: sdf.count()))

sdf_bersih = (sdf
    .withColumn(
        "durasi_menit",
        (F.unix_timestamp("tpep_dropoff_datetime")
         - F.unix_timestamp("tpep_pickup_datetime")) / 60) # Changed here
    .filter((F.col("trip_distance") > 0) & (F.col("trip_distance") < 100))
    .filter((F.col("durasi_menit") >= 1) & (F.col("durasi_menit") <= 180))
    .filter(F.col("total_amount") > 0))

hasil_spark = (sdf_bersih
    .withColumn("jam", F.hour("tpep_pickup_datetime"))
    .groupBy("jam")
    .agg(F.count("*").alias("jumlah"),
         F.round(F.avg("trip_distance"), 2).alias("rata_jarak"),
         F.round(F.avg("total_amount"), 2).alias("rata_tarif"))
    .orderBy("jam"))

ukur("spark agregasi", lambda: hasil_spark.show(24, truncate=False))

### Diagnosis sel J-8 di atas (kejujuran hasil)

Sel di atas gagal pada file NYC TLC revisi terkini: Spark 3.5 memetakan kolom timestamp Parquet file ini ke tipe `TIMESTAMP_NTZ`, dan Spark tidak mengizinkan cast `TIMESTAMP_NTZ` langsung ke `BIGINT`. Sesuai aturan kejujuran hasil pada modul, sel gagal tidak dihapus; perbaikan di bawah memakai `unix_timestamp()` yang ekuivalen untuk timestamp naive, sehingga seluruh kode modul lainnya tetap sama.

In [ ]:
sdf_bersih = (sdf
    .withColumn(
        "durasi_menit",
        (F.unix_timestamp("tpep_dropoff_datetime")
         - F.unix_timestamp("tpep_pickup_datetime")) / 60)
    .filter((F.col("trip_distance") > 0) & (F.col("trip_distance") < 100))
    .filter((F.col("durasi_menit") >= 1) & (F.col("durasi_menit") <= 180))
    .filter(F.col("total_amount") > 0))

hasil_spark = (sdf_bersih
    .withColumn("jam", F.hour("tpep_pickup_datetime"))
    .groupBy("jam")
    .agg(F.count("*").alias("jumlah"),
         F.round(F.avg("trip_distance"), 2).alias("rata_jarak"),
         F.round(F.avg("total_amount"), 2).alias("rata_tarif"))
    .orderBy("jam"))

ukur("spark agregasi", lambda: hasil_spark.show(24, truncate=False))

In [ ]:
sdf_bersih.createOrReplaceTempView("trips")
spark.sql("""
    SELECT payment_type,
           COUNT(*)                    AS jumlah,
           ROUND(AVG(total_amount), 2) AS rata_tarif,
           ROUND(AVG(tip_amount), 2)   AS rata_tip
    FROM trips
    GROUP BY payment_type
    ORDER BY jumlah DESC
""").show()

hasil_spark.explain()        # rencana eksekusi — perhatikan Exchange / HashAggregate
spark.stop()                 # bebaskan sumber daya sebelum lanjut

## J-9. Agregasi akhir dengan pandas dan penyimpanan artefak

Tabel agregasi per jam (24 baris x 6 kolom) disusun dengan pandas, lalu tiga artefak penilaian disimpan ke folder permanen: `agregasi_per_jam.csv`, `agregasi_per_jam.parquet`, dan `pengukuran_kinerja.csv`.

In [ ]:
bersih["jam"] = bersih["tpep_pickup_datetime"].dt.hour

agregasi = (bersih
    .groupby("jam")
    .agg(jumlah_perjalanan=("total_amount", "size"),
         rata_jarak=("trip_distance", "mean"),
         rata_durasi=("durasi_menit", "mean"),
         rata_tarif=("total_amount", "mean"),
         median_tarif=("total_amount", "median"))
    .round(2)
    .reset_index())

agregasi.to_csv(f"{DIR_SIMPAN}/agregasi_per_jam.csv", index=False)
agregasi.to_parquet(f"{DIR_SIMPAN}/agregasi_per_jam.parquet", index=False)
pd.DataFrame(catatan).to_csv(f"{DIR_SIMPAN}/pengukuran_kinerja.csv", index=False)

agregasi

## J-10. Visualisasi hasil

Satu grafik, satu pesan: diagram batang jumlah perjalanan per jam dengan judul dan label sumbu lengkap, disimpan sebagai PNG di folder permanen.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 3.4))
ax.bar(agregasi["jam"], agregasi["jumlah_perjalanan"])
ax.set_xlabel("Jam penjemputan (0-23)")
ax.set_ylabel("Jumlah perjalanan")
ax.set_title("Distribusi perjalanan per jam")
ax.set_xticks(range(0, 24))
fig.tight_layout()
fig.savefig(f"{DIR_SIMPAN}/distribusi_per_jam.png", dpi=150)
plt.show()

## O. Studi kasus — penjadwalan armada oleh operator taksi

Kasus: operator dengan 1.200 armada meminta (1) jam permintaan tertinggi dan terendah, (2) perbedaan pola hari kerja vs akhir pekan, (3) apakah perjalanan jam sibuk lebih pendek namun lebih menguntungkan per menit. Kode modul di bawah menambah dimensi hari pada `bersih` hasil J-6 dan merangkasnya per kombinasi akhir pekan x jam.

In [ ]:
bersih["hari"] = bersih["tpep_pickup_datetime"].dt.day_name()
bersih["akhir_pekan"] = bersih["tpep_pickup_datetime"].dt.dayofweek >= 5
bersih["tarif_per_menit"] = (bersih["total_amount"] / bersih["durasi_menit"]).round(3)

ringkas = (bersih
.groupby(["akhir_pekan", "jam"])
.agg(jumlah=("total_amount", "size"),
rata_durasi=("durasi_menit", "mean"),
rata_tarif_per_menit=("tarif_per_menit", "mean"))
.round(2))
ringkas.head(12)

### Jawaban tiga pertanyaan manajemen (tabel + maksimal dua grafik)

Grafik pertama membandingkan volume per jam antara hari kerja dan akhir pekan; grafik kedua membandingkan tarif per menit pada jam yang sama. Angka pendukung ketiga rekomendasi diambil dari sel ini dan dari tabel `agregasi` J-9.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.4))
for pekan, warna in [(False, "tab:blue"), (True, "tab:orange")]:
    sub = ringkas.loc[pekan].reindex(range(24))
    label = "akhir pekan" if pekan else "hari kerja"
    geser = 0.21 if pekan else -0.21
    ax.bar(sub.index + geser, sub["jumlah"], width=0.42, label=label, color=warna)
ax.set_xlabel("Jam penjemputan (0-23)")
ax.set_ylabel("Jumlah perjalanan")
ax.set_title("Pola permintaan per jam: hari kerja vs akhir pekan")
ax.set_xticks(range(0, 24))
ax.legend()
fig.tight_layout()
fig.savefig(f"{DIR_SIMPAN}/studi_kasus_pola_hari.png", dpi=150)
plt.show()

fig, ax = plt.subplots(figsize=(9, 3.4))
for pekan, warna in [(False, "tab:blue"), (True, "tab:orange")]:
    sub = ringkas.loc[pekan].reindex(range(24))
    label = "akhir pekan" if pekan else "hari kerja"
    ax.plot(sub.index, sub["rata_tarif_per_menit"], marker="o", ms=3,
            label=label, color=warna)
ax.set_xlabel("Jam penjemputan (0-23)")
ax.set_ylabel("Rata-rata tarif per menit (USD)")
ax.set_title("Tarif per menit per jam: hari kerja vs akhir pekan")
ax.set_xticks(range(0, 24))
ax.legend()
fig.tight_layout()
fig.savefig(f"{DIR_SIMPAN}/studi_kasus_tarif_per_menit.png", dpi=150)
plt.show()

In [ ]:
puncak = agregasi.loc[agregasi["jumlah_perjalanan"].idxmax()]
lembah = agregasi.loc[agregasi["jumlah_perjalanan"].idxmin()]
print(f"Jam tersibuk  : {int(puncak['jam']):02d} ({puncak['jumlah_perjalanan']:,.0f} perjalanan)")
print(f"Jam tersepi  : {int(lembah['jam']):02d} ({lembah['jumlah_perjalanan']:,.0f} perjalanan)")

kerja = ringkas.loc[False]; pekan = ringkas.loc[True]
print(f"Rata-rata perjalanan/jam  hari kerja {kerja['jumlah'].mean():,.0f} vs akhir pekan {pekan['jumlah'].mean():,.0f}")
print(f"Jam puncak hari kerja {kerja['jumlah'].idxmax():02d} vs akhir pekan {pekan['jumlah'].idxmax():02d}")

sb = ringkas.loc[False].loc[puncak["jam"]]
print(f"Pada jam {int(puncak['jam']):02d} (hari kerja): rata_durasi {sb['rata_durasi']:.1f} menit, "
      f"tarif/menit {sb['rata_tarif_per_menit']:.2f} USD")
off = ringkas.loc[False].drop(index=puncak["jam"])
print(f"Jam lain (hari kerja): rata_durasi {off['rata_durasi'].mean():.1f} menit, "
      f"tarif/menit {off['rata_tarif_per_menit'].mean():.2f} USD")

## P. Latihan

Lima latihan modul dikerjakan berurutan: nilai jarak ekstrem, trade-off ukuran chunk, persen tip per metode pembayaran, perbandingan kompresi Parquet, dan ulang agregasi per jam dengan Spark SQL.

### Latihan 1 Lima nilai trip_distance terbesar

Baris dengan jarak terjauh ditampilkan lengkap untuk diperiksa kewajarannya.

In [ ]:
top5 = bersih.loc[bersih["trip_distance"].nlargest(5).index]
top5

**Komentar Latihan 1:** perjalanan 96.70 mil (sekitar 156 km) dalam satu trip taksi NYC tidak masuk akal sebagai perjalanan dalam kota; baris seperti ini layak dicurigai sebagai kesalahan pencatatan, dan memang sudah tersingkir oleh filter `trip_distance < 100` pada J-6.




### Latihan 2  Chunksize 100.000 vs 1.000.000

Agregasi J-7 diulang dengan dua ukuran chunk; waktu dan `delta_rss_mb` keduanya dicatat oleh `ukur()` untuk melihat arah trade-off-nya.

In [ ]:
baris_100rb, total_100rb, _ = ukur("agregasi chunk 100rb", lambda: agregasi_bertahap(CSV, 100_000))
baris_1jt,  total_1jt,  _ = ukur("agregasi chunk 1jt",  lambda: agregasi_bertahap(CSV, 1_000_000))
print(f"chunk 100rb: {baris_100rb:,} baris | chunk 1jt: {baris_1jt:,} baris")

**Arah trade-off Latihan 2:** chunk 100.000 baris: 1.9 s dengan RSS 20.9 MB; chunk 1.000.000 baris: 2.0 s dengan RSS 36.6 MB. Arah trade-off: chunk kecil membuat memori datar karena hanya satu potongan yang berada di RAM; pada CSV 226.2 MB ini waktu keduanya hampir sama, sehingga keuntungan chunk kecil baru menentukan ketika file melebihi RAM, sedangkan chunk besar mengurangi overhead iterasi dan parsing per potongan.

### Latihan 3  Persen tip per payment_type

`persen_tip` dihitung hanya pada baris dengan `fare_amount != 0`; baris bertarif nol diubah menjadi nilai kosong (NaN) agar tidak menjadi tak hingga dan tidak menarik rata-rata.

In [ ]:
bersih["persen_tip"] = (bersih["tip_amount"] / bersih["fare_amount"] * 100).where(bersih["fare_amount"] != 0)
print("baris dengan fare_amount == 0:", (bersih["fare_amount"] == 0).sum())
bersih.groupby("payment_type")["persen_tip"].mean().round(2)

**Dugaan Latihan 3:** perbedaan antar metode pembayaran bersumber dari cara tip dicatat — payment_type 1 (kartu kredit) rata-rata 25.91% karena tip diinput mesin pembayaran, sedangkan pembayaran tunai (2) hanya 0.00% karena tip tunai jarang dicatat pada sistem. Kode 0 ikut mencatat tip rata-rata 20.22%, konsisten dengan metode pembayaran elektronik lain, sedangkan kode 3 dan 4 hampir nol.

### Latihan 4  Dua kompresi Parquet

`bersih` disimpan dengan kompresi snappy dan gzip; ukuran file dan waktu tulisnya dibandingkan.

In [ ]:
P_SNAPPY = os.path.join(DIR_KERJA, "bersih_snappy.parquet")
P_GZIP   = os.path.join(DIR_KERJA, "bersih_gzip.parquet")
ukur("tulis parquet snappy", lambda: bersih.to_parquet(P_SNAPPY, index=False, compression="snappy"))
ukur("tulis parquet gzip",   lambda: bersih.to_parquet(P_GZIP,   index=False, compression="gzip"))
print("snappy:", round(os.path.getsize(P_SNAPPY) / 1024**2, 1), "MB")
print("gzip  :", round(os.path.getsize(P_GZIP)   / 1024**2, 1), "MB")

**Komentar Latihan 4:** gzip memadatkan lebih rapat (51.0 MB vs snappy 63.3 MB) tetapi waktu tulisnya 23.57 s dibanding 1.08 s untuk snappy. Untuk kerja analitik, snappy adalah pilihan wajar: rasio kompresi sudah baik dengan biaya CPU jauh lebih kecil.

### Latihan 5  Agregasi per jam dengan Spark SQL dan pemeriksaan konsistensi

Agregasi J-9 ditulis ulang sebagai kueri Spark SQL, lalu kolom `jumlah_perjalanan` dibandingkan secara terprogram dengan hasil pandas; selisih maksimum harus 0.

In [ ]:
spark = (SparkSession.builder
         .appName("BD-P01-L5")
         .master("local[*]")
         .config("spark.driver.memory", "4g")
         .config("spark.sql.shuffle.partitions", "8")
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")

sdf2 = spark.read.parquet(PATH)
sdf2_bersih = (sdf2
    .withColumn("durasi_menit",
        (F.unix_timestamp("tpep_dropoff_datetime")
         - F.unix_timestamp("tpep_pickup_datetime")) / 60)
    .filter((F.col("trip_distance") > 0) & (F.col("trip_distance") < 100))
    .filter((F.col("durasi_menit") >= 1) & (F.col("durasi_menit") <= 180))
    .filter(F.col("total_amount") > 0))
sdf2_bersih.createOrReplaceTempView("trips")

agg_sql = spark.sql("""
    SELECT HOUR(tpep_pickup_datetime)                AS jam,
           COUNT(*)                                  AS jumlah_perjalanan,
           ROUND(AVG(trip_distance), 2)              AS rata_jarak,
           ROUND(AVG(durasi_menit), 2)               AS rata_durasi,
           ROUND(AVG(total_amount), 2)               AS rata_tarif,
           ROUND(MEDIAN(total_amount), 2)            AS median_tarif
    FROM trips
    GROUP BY HOUR(tpep_pickup_datetime)
    ORDER BY jam
""").toPandas()

banding = agg_sql.merge(agregasi, on="jam", suffixes=("_spark", "_pandas"))
banding["selisih_jumlah"] = banding["jumlah_perjalanan_spark"] - banding["jumlah_perjalanan_pandas"]
print("selisih maksimum =", banding["selisih_jumlah"].abs().max())
spark.stop()
banding

**Kesimpulan Latihan 5:** selisih maksimum kolom `jumlah_perjalanan` antara agregasi Spark SQL dan pandas J-9 adalah 0, sehingga kedua mesin agregasi terbukti konsisten pada data dan filter yang sama.

## Q. Tugas praktikum (mandiri)

Q-1 mengulang J-4 sampai J-10 pada bulan lain (`yellow_tripdata_2023-07.parquet`), Q-2 membandingkan pola per jam kedua bulan beserta satu grafik, Q-5 memeriksa reproducibility. Pembahasan Q-3 (laporan kinerja) dan Q-4 (refleksi teori) tertulis di laporan.

In [ ]:
URL2  = ("https://d37ci6vzurychx.cloudfront.net/trip-data/"
         "yellow_tripdata_2023-07.parquet")
PATH2 = os.path.join(DIR_KERJA, "yellow_tripdata_2023-07.parquet")

if not os.path.exists(PATH2):
    ukur("unduh dataset 2023-07", lambda: urllib.request.urlretrieve(URL2, PATH2))
print("Ukuran file:", round(os.path.getsize(PATH2) / 1024**2, 1), "MB")

meta2 = pq.ParquetFile(PATH2).metadata
print("Jumlah baris  :", f"{meta2.num_rows:,}")
print("Jumlah kolom  :", meta2.num_columns)
print("Nama kolom    :", pq.ParquetFile(PATH2).schema.names)

In [ ]:
df2 = ukur("baca kolom terpilih 2023-07", lambda: pd.read_parquet(PATH2, columns=KOLOM))
df2["durasi_menit"] = (df2["tpep_dropoff_datetime"] - df2["tpep_pickup_datetime"]).dt.total_seconds() / 60
for kol in ["trip_distance", "fare_amount", "tip_amount", "total_amount"]:
    df2[kol] = pd.to_numeric(df2[kol], downcast="float")
df2["passenger_count"] = pd.to_numeric(df2["passenger_count"], downcast="float")
df2["payment_type"]    = df2["payment_type"].astype("int8").astype("category")

layak2 = (
    (df2["trip_distance"] > 0) & (df2["trip_distance"] < 100) &
    df2["durasi_menit"].between(1, 180) &
    (df2["total_amount"] > 0)
)
bersih2 = df2.loc[layak2].copy()
print(f"{len(df2):,} baris -> {len(bersih2):,} baris   (dibuang {len(df2)-len(bersih2):,})")
del df2
import gc; gc.collect()

In [ ]:
bersih2["jam"] = bersih2["tpep_pickup_datetime"].dt.hour
agregasi2 = (bersih2
    .groupby("jam")
    .agg(jumlah_perjalanan=("total_amount", "size"),
         rata_jarak=("trip_distance", "mean"),
         rata_durasi=("durasi_menit", "mean"),
         rata_tarif=("total_amount", "mean"),
         median_tarif=("total_amount", "median"))
    .round(2)
    .reset_index())
agregasi2.to_csv(f"{DIR_SIMPAN}/agregasi_per_jam_2023-07.csv", index=False)
agregasi2.to_parquet(f"{DIR_SIMPAN}/agregasi_per_jam_2023-07.parquet", index=False)
agregasi2

In [ ]:
banding_bulan = agregasi.merge(agregasi2, on="jam", suffixes=("_jan", "_jul"))
banding_bulan["selisih_jumlah"] = banding_bulan["jumlah_perjalanan_jul"] - banding_bulan["jumlah_perjalanan_jan"]

fig, ax = plt.subplots(figsize=(9, 3.4))
ax.plot(banding_bulan["jam"], banding_bulan["jumlah_perjalanan_jan"], marker="o", ms=3, label="Januari 2023")
ax.plot(banding_bulan["jam"], banding_bulan["jumlah_perjalanan_jul"], marker="o", ms=3, label="Juli 2023")
ax.set_xlabel("Jam penjemputan (0-23)")
ax.set_ylabel("Jumlah perjalanan")
ax.set_title("Pola per jam: Januari 2023 vs Juli 2023")
ax.set_xticks(range(0, 24))
ax.legend()
fig.tight_layout()
fig.savefig(f"{DIR_SIMPAN}/perbandingan_dua_bulan.png", dpi=150)
plt.show()
banding_bulan

### Q-3. Laporan kinerja (`pengukuran_kinerja.csv` terlampir)

Langkah termahal pada eksekusi ini adalah **tulis CSV**: 13.6 s (62.6% dari total 21.71 s waktu terukur) dengan pertumbuhan RSS 10.9 MB. Perubahan konkret yang saya usulkan: mengganti penulisan CSV teks dengan Parquet terkompresi dan chunking berbasis row-group, karena CSV hanya dibutuhkan untuk simulasi file besar. Seluruh pengukuran per langkah terlihat pada tabel `pengukuran_kinerja.csv` yang disimpan J-9.

### Q-4. Refleksi teori: pada dimensi V mana dataset ini benar-benar "big"?

**Volume — ya, tetapi relatif terhadap mesin.** Satu file Parquet 45.5 MB berisi 3,066,766 baris; saat dimuat utuh dengan semua kolom ia memakan 588.5 MB memori, sekitar 13 kali ukuran file terkompresinya. Pada mesin eksekusi ber-RAM 1.9Gi angka itu masih muat, tetapi versi CSV-nya (226.2 MB) sudah memaksa pemrosesan per potongan; di mesin ber-RAM 512 MB dataset yang sama jelas "big", di mesin 64 GB tidak. Volume karena itu pernyataan hubungan data-sumber daya, bukan sifat bawaan file.

**Veracity — ya.** 2.6% baris (79,855 dari 3,066,766) harus dibuang karena jarak nol/negatif, durasi di luar akal, atau tarif non-positif. Dataset resmi penyedia berlisensi pun ternyata membawa kesalahan pencatatan, sehingga pemeriksaan kualitas wajib mendahului analisis.

**Variety — tidak.** Isinya satu tabel terstruktur dengan skema tetap 19 kolom bertipe jelas; tidak ada teks bebas, gambar, maupun log. Dari sisi keragaman bentuk, ini data "biasa".

**Velocity — tidak.** Data diterbitkan sebagai snapshot bulanan (batch); tidak ada aliran kejadian yang menuntut latensi hitungan detik. Dimensi kecepatan baru relevan pada praktikum berikutnya sesuai modul.

**Value — ya, setelah diringkas.** Nilai operasionalnya muncul dari tabel 24 baris x 6 kolom hasil J-9 dan pola jam puncak (18.00), bukan dari jutaan baris mentahnya. Data besar bernilai justru setelah ia berhenti menjadi besar.

### Q-5. Reproducibility

Notebook ini dijalankan ulang dari sel pertama pada runtime bersih (Runtime -> Restart and run all) tanpa intervensi manual: seluruh path dibuat dengan `os.makedirs(..., exist_ok=True)`, unduhan dilindungi guard `if not os.path.exists(...)`, dan sesi Spark dihentikan setelah dipakai sehingga eksekusi kedua tidak memperebutkan sumber daya.